# 01 - Data loading and quality

Loads the Online Retail II dataset (both sheets) and reports data quality.

**No rows are dropped or modified in this notebook.** It only describes the raw data so cleaning decisions can be made explicitly in the next step.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.load_data import RAW_PATH, PROCESSED_PATH, load_raw, save_processed, load_processed

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. Load

The Excel read is slow (a few minutes). It runs only when the parquet cache does not exist yet. Delete `data/processed/online_retail_raw.parquet` to force a fresh read.

In [2]:
if not PROCESSED_PATH.exists():
    print(f"Reading Excel (slow): {RAW_PATH}")
    save_processed(load_raw(RAW_PATH), PROCESSED_PATH)
    print(f"Saved parquet: {PROCESSED_PATH}")

df = load_processed(PROCESSED_PATH)
df.head()

Reading Excel (slow): E:\BI\mini\online-retail-analysis\data\raw\online_retail_II.xlsx


Saved parquet: E:\BI\mini\online-retail-analysis\data\processed\online_retail_raw.parquet


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010


## 2. Data quality summary

In [3]:
summary: list[tuple[str, str, object]] = []   # (section, metric, value)

def add(section: str, metric: str, value: object) -> None:
    summary.append((section, metric, value))

def pct(n: int, total: int = len(df)) -> str:
    return f"{n / total:.2%}"

### Shape, dtypes, memory

In [4]:
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
add("Overview", "Rows", f"{len(df):,}")
add("Overview", "Columns", df.shape[1])
add("Overview", "Memory usage (deep)", f"{mem_mb:.1f} MB")
for sheet, n in df["source_sheet"].value_counts(sort=False).items():
    add("Overview", f"Rows from sheet '{sheet}'", f"{n:,}")

print("Shape:", df.shape)
print(f"Memory usage: {mem_mb:.1f} MB\n")
print(df.dtypes)

Shape: (1067371, 9)
Memory usage: 119.3 MB

Invoice                 string
StockCode               string
Description             string
Quantity                 int64
InvoiceDate     datetime64[us]
Price                  float64
Customer ID              Int64
Country                 string
source_sheet          category
dtype: object


### Missing values per column

In [5]:
missing = pd.DataFrame({"missing": df.isna().sum(), "pct": (df.isna().mean() * 100).round(2)})
for col, row in missing.iterrows():
    add("Missing values", col, f"{int(row['missing']):,} ({row['pct']:.2f}%)")
missing

,missing,pct
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00
source_sheet,0,0.00


### Duplicates, negative quantities, non-positive prices, cancellations

In [6]:
n_dup = int(df.duplicated().sum())
n_dup_excl_sheet = int(df.drop(columns="source_sheet").duplicated().sum())
qty_neg = df["Quantity"] < 0
price_le0 = df["Price"] <= 0
is_cancel = df["Invoice"].str.upper().str.startswith("C")

checks = {
    "Fully duplicated rows": n_dup,
    "Duplicated rows ignoring source_sheet (incl. overlap between sheets)": n_dup_excl_sheet,
    "Rows with Quantity < 0": int(qty_neg.sum()),
    "Rows with Price <= 0": int(price_le0.sum()),
    "  of which Price == 0": int((df["Price"] == 0).sum()),
    "  of which Price < 0": int((df["Price"] < 0).sum()),
    "Cancellation rows (Invoice starts with 'C')": int(is_cancel.sum()),
    "  cancellations with Quantity < 0": int((is_cancel & qty_neg).sum()),
    "  cancellations with Quantity >= 0": int((is_cancel & ~qty_neg).sum()),
    "Quantity < 0 but NOT a cancellation": int((~is_cancel & qty_neg).sum()),
    "  of those, Customer ID missing": int((~is_cancel & qty_neg & df["Customer ID"].isna()).sum()),
}
for k, v in checks.items():
    add("Row-level issues", k.strip(), f"{v:,} ({pct(v)})")
    print(f"{k:<70} {v:>10,}  ({pct(v)})")

n_cancel_inv = df.loc[is_cancel, "Invoice"].nunique()
add("Row-level issues", "Unique cancellation invoices", f"{n_cancel_inv:,}")
print(f"{'Unique cancellation invoices':<70} {n_cancel_inv:>10,}")

Fully duplicated rows                                                      12,133  (1.14%)
Duplicated rows ignoring source_sheet (incl. overlap between sheets)       34,335  (3.22%)
Rows with Quantity < 0                                                     22,950  (2.15%)
Rows with Price <= 0                                                        6,207  (0.58%)
  of which Price == 0                                                       6,202  (0.58%)
  of which Price < 0                                                            5  (0.00%)
Cancellation rows (Invoice starts with 'C')                                19,494  (1.83%)
  cancellations with Quantity < 0                                          19,493  (1.83%)
  cancellations with Quantity >= 0                                              1  (0.00%)
Quantity < 0 but NOT a cancellation                                         3,457  (0.32%)
  of those, Customer ID missing                                             3,457  (0.32%)

In [7]:
# Invoices that are neither plain numbers nor 'C' + number (e.g. 'A' = adjust bad debt)
odd_mask = ~df["Invoice"].str.fullmatch(r"C?\d+")
add("Row-level issues", "Rows whose Invoice is not numeric or 'C'+numeric", f"{int(odd_mask.sum()):,}")
print("Rows with non-standard invoice numbers:", int(odd_mask.sum()))
df[odd_mask]

Rows with non-standard invoice numbers: 6


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,<NA>,United Kingdom,Year 2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,<NA>,United Kingdom,Year 2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,<NA>,United Kingdom,Year 2009-2010
825443,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,<NA>,United Kingdom,Year 2010-2011
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom,Year 2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom,Year 2010-2011


### Date range and rows per month

In [8]:
add("Dates", "Min InvoiceDate", df["InvoiceDate"].min())
add("Dates", "Max InvoiceDate", df["InvoiceDate"].max())
for sheet, g in df.groupby("source_sheet", observed=True):
    add("Dates", f"Range in sheet '{sheet}'", f"{g['InvoiceDate'].min()} to {g['InvoiceDate'].max()}")

rows_per_month = df.groupby(df["InvoiceDate"].dt.to_period("M")).size().rename("rows")
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
rows_per_month.to_frame()

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


,rows
InvoiceDate,
2009-12,45228
2010-01,31555
2010-02,29388
2010-03,41511
2010-04,34057
2010-05,35323
2010-06,39983
2010-07,33383
2010-08,33306


### Cardinalities and top countries

In [9]:
uniques = {
    "Unique customers (non-null Customer ID)": df["Customer ID"].nunique(),
    "Unique invoices": df["Invoice"].nunique(),
    "Unique StockCodes": df["StockCode"].nunique(),
    "Unique Descriptions": df["Description"].nunique(),
    "Unique countries": df["Country"].nunique(),
}
for k, v in uniques.items():
    add("Unique counts", k, f"{v:,}")
    print(f"{k:<45} {v:>8,}")

top_countries = df["Country"].value_counts().head(10).rename("rows").to_frame()
top_countries["pct"] = (top_countries["rows"] / len(df) * 100).round(2)
top_countries

Unique customers (non-null Customer ID)          5,942
Unique invoices                                 53,628
Unique StockCodes                                5,305
Unique Descriptions                              5,698
Unique countries                                    43


,rows,pct
Country,,
United Kingdom,981330,91.94
EIRE,17866,1.67
Germany,17624,1.65
France,14330,1.34
Netherlands,5140,0.48
Spain,3811,0.36
Switzerland,3189,0.3
Belgium,3123,0.29
Portugal,2620,0.25


### Non-product StockCodes

Regular product codes start with 5 digits, optionally followed by letters (e.g. `85123A`). Everything else is listed below with its row count.

In [10]:
codes = df["StockCode"].astype(str)
is_product_code = codes.str.match(r"^\d{5}")
non_product = (
    df.loc[~is_product_code]
      .groupby("StockCode")
      .agg(rows=("StockCode", "size"),
           example_description=("Description", lambda s: s.dropna().iloc[0] if s.notna().any() else None),
           total_quantity=("Quantity", "sum"))
      .sort_values("rows", ascending=False)
)
n_np_rows = int(non_product["rows"].sum())
add("StockCodes", "Distinct non-product StockCodes", f"{len(non_product):,}")
add("StockCodes", "Rows with non-product StockCodes", f"{n_np_rows:,} ({pct(n_np_rows)})")

# Product codes whose letter suffix differs only by case (e.g. 85123A vs 85123a)
pc = codes[is_product_code]
case_variants = pc.groupby(pc.str.upper()).nunique()
add("StockCodes", "Product codes with upper/lower-case variants", f"{int((case_variants > 1).sum()):,}")

# StockCodes mapped to more than one Description
desc_per_code = df.dropna(subset=["Description"]).groupby("StockCode")["Description"].nunique()
add("StockCodes", "StockCodes with more than one Description", f"{int((desc_per_code > 1).sum()):,}")

pd.set_option("display.max_rows", 200)
non_product

,rows,example_description,total_quantity
StockCode,,,
POST,2122,POSTAGE,10108
DOT,1446,DOTCOM POSTAGE,2938
M,1421,Manual,4607
C2,282,CARRIAGE,716
D,177,Discount,-2872
S,104,SAMPLES,-98
BANK CHARGES,102,Bank Charges,-42
ADJUST,67,Adjustment by john on 26/01/2010 16,5
AMAZONFEE,43,AMAZON FEE,-35


### Value ranges (for outlier awareness)

In [11]:
for col in ["Quantity", "Price"]:
    add("Value ranges", f"{col} min / median / max",
        f"{df[col].min():,.2f} / {df[col].median():,.2f} / {df[col].max():,.2f}")
df[["Quantity", "Price"]].describe(percentiles=[0.01, 0.5, 0.99]).T

,count,mean,std,min,1%,50%,99%,max
Quantity,1067371.0,9.938898,172.705794,-80995.00,-3.00,3.0,100.0,80995.0
Price,1067371.0,4.649388,123.553059,-53594.36,0.21,2.1,18.0,38970.0


## 3. Save report to `reports/data_quality.md`

In [12]:
def md_table(frame: pd.DataFrame) -> str:
    cols = list(frame.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(str(v).replace("|", "/") for v in row) + " |"
              for row in frame.itertuples(index=False)]
    return "\n".join(lines)

summary_df = pd.DataFrame(summary, columns=["Section", "Metric", "Value"])
parts = [
    "# Data Quality Report: Online Retail II (raw, uncleaned)",
    f"Generated by `notebooks/01_data_loading_and_quality.ipynb`. "
    "No rows have been removed or modified.",
    "## Summary",
    md_table(summary_df),
    "## Rows per month",
    md_table(rows_per_month.reset_index().rename(columns={"InvoiceDate": "Month"})),
    "## Top 10 countries by row count",
    md_table(top_countries.reset_index()),
    "## Non-product StockCodes",
    md_table(non_product.reset_index()),
]
report_path = ROOT / "reports" / "data_quality.md"
report_path.write_text("\n\n".join(parts) + "\n", encoding="utf-8")
print(f"Wrote {report_path}")
summary_df

Wrote E:\BI\mini\online-retail-analysis\reports\data_quality.md


,Section,Metric,Value
0,Overview,Rows,"1,067,371"
1,Overview,Columns,9
2,Overview,Memory usage (deep),119.3 MB
3,Overview,Rows from sheet 'Year 2009-2010',"525,461"
4,Overview,Rows from sheet 'Year 2010-2011',"541,910"
5,Missing values,Invoice,0 (0.00%)
6,Missing values,StockCode,0 (0.00%)
7,Missing values,Description,"4,382 (0.41%)"
8,Missing values,Quantity,0 (0.00%)
9,Missing values,InvoiceDate,0 (0.00%)
